# 07. Согласие аннотаторов: первый пилот

Два независимых экспорта Label Studio для каждого набора. Метрики рассчитываются до сверки по `src/agreement.py`. Gold после сверки строится отдельно через `src/build_gold.py`.

Пилот: 60 SMS и 30 изображений. Для SMS единица согласия — сообщение, для Cars — рамка автомобиля; две рамки одного автомобиля образуют одну пару. Основная разметка в расчёт не входит.

In [1]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / "reports" / "agreement").is_dir():
    root = root.parent

def read(path):
    return pd.read_csv(root / path)

base = "reports/agreement"
sms = read(f"{base}/sms_items.csv")
cars = read(f"{base}/cars_items.csv")
sms_gold = read("data/annotations/sms_gold_pilot.csv")
cars_gold = read("data/annotations/cars_gold_pilot.csv")

## SMS

PA — доля совпавших меток; каппа Коэна вычитает ожидаемое случайное совпадение по долям классов у A и B. Доверительный интервал получен бутстрэпом пар сообщений (2 000 повторов, seed 42). PA подтипов рассчитывается только для сообщений, которым оба поставили `spam`. `uncertain` учитывается отдельно от метки.

In [2]:
from IPython.display import display

metric = read(f"{base}/sms_metrics.csv").set_index("Unnamed: 0")
metric.index.name = "Метрика"
display(metric)
display(pd.crosstab(sms.label_A, sms.label_B))
display(sms_gold.groupby(["label", "resolution"]).size().unstack(fill_value=0))

,значение
Метрика,
"объектов, размеченных обоими",60
Percentage Agreement (метка),0.9667
Cohen's kappa (метка),0.9306
"kappa, 95 % ДИ (бутстрэп)",0.820–1.000
Krippendorff alpha (метка),0.9311
PA подтипа (оба spam),0.8696
"доля uncertain, A / B",0.017 / 0.083
"медиана времени, с (A / B)",6.4 / 10.1
"суммарное время, мин (A / B)",19.4 / 23.1


label_B,ham,spam
label_A,,
ham,35,2
spam,0,23


resolution,agreed,resolved
label,,
ham,35,2
spam,19,3


Согласие по классам — 58/60, по подтипам — 20/23. Пороги таблицы 13 выполнены. В сверке рассмотрено 11 случаев. `sms_new_0100` остаётся в независимом расчёте и исключается из gold до пересмотра (codebook v1.2, § 9.2).

## Cars

IoU — отношение площади пересечения рамок к площади их объединения. Рамки сопоставляются внутри каждого изображения один к одному при IoU ≥ 0,5. F1 согласия = 2 × число пар / суммарное число рамок A и B. Медиана IoU и согласие по атрибутам описывают только сопоставленные пары. Расхождение числа рамок — модуль разницы общих количеств, делённый на их полусумму.

In [3]:
metric = read(f"{base}/cars_metrics.csv").set_index("Unnamed: 0")
metric.index.name = "Метрика"
display(metric)
review = read("labeling/resolution/cars_resolution.csv")
display(pd.crosstab(review.status, review.keep).rename(columns={0: "Удалить", 1: "Оставить"}))
print(f"Gold: {len(cars_gold)} рамок на {cars_gold.image_id.nunique()} изображениях; "
      f"depiction: {cars_gold.depiction.sum()}")
counts = cars[["id", "boxes_A", "boxes_B", "matched"]].drop_duplicates()
counts = counts.assign(разница=(counts.boxes_A - counts.boxes_B).abs())
display(counts.sort_values("разница", ascending=False).head(5).reset_index(drop=True))

,значение
Метрика,
"изображений, размеченных обоими",30
рамок A / B,157 / 101
"сопоставлено пар (IoU ≥ 0,5)",93
F1 согласия по объектам = 2·m / (nA + nB),0.7209
медианный IoU сопоставленных,0.9023
средний IoU сопоставленных,0.8783
"доля пар с IoU ≥ 0,8",0.8065
"расхождение по числу объектов, %",43.41
изображений с разным числом рамок,17


keep,Удалить,Оставить
status,,
matched,0,29
only_A,48,16
only_B,2,6


Gold: 115 рамок на 30 изображениях; depiction: 1


,id,boxes_A,boxes_B,matched,разница
0,img_new_0023,17,6,5,11
1,img_new_0071,9,0,0,9
2,img_new_0033,12,5,4,7
3,img_new_0034,13,6,5,7
4,img_new_0038,10,5,5,5


Медианный IoU сопоставленных рамок — 0,902. Пороги F1 и расхождения количества не выполнены. Сверка охватила 101 строку: 72 одиночные рамки, 18 пар с IoU < 0,8, ещё 11 пар с различиями по атрибутам или флагом сомнения на изображении. Все 29 спорных пар оставлены, итог содержит 93 парные и 22 одиночные рамки. У сопоставленных рамок сборщик объединяет атрибуты по правилу ИЛИ; выбор `use` относится к координатам. Gold не используется для повторного расчёта независимого согласия.

Время приведено по исходному `lead_time`. Границы сессий в выгрузках отсутствуют, поэтому первые пять задач каждой сессии не исключены. У каждого аннотатора Cars две задачи длительностью более 10 минут; суммарное время нельзя трактовать как чистое рабочее время.

## Следующий этап

- SMS: оба аннотатора размечают оставшиеся 90 сообщений по v1.2.
- Cars: повторный пилот — 10 общих кадров из оставшихся 24. После достижения порогов — остальные 14. Эти 10 кадров в основную независимую выборку не включаются повторно.
- Метрики последующих этапов считаются отдельно. При сборке итоговых наборов используется одна согласованная версия каждого ID; рамки `depiction` из обучающей выборки исключаются. Пилотный EDA и предварительные оценки по меткам источника не заменяют сравнение вариантов обучения на фиксированных разбиениях.